In [2]:
import pandas as pd

# 1. Load the files
# Replace filenames if yours are named differently
weseq_clin = pd.read_csv('weseq_286_Clinical.csv')
weseq_mut = pd.read_csv('weseq_286_Mutation.csv')

# These are the 20 genes found in the TCGA dataset
genes_tcga = [
    'IDH1', 'TP53', 'ATRX', 'PTEN', 'EGFR', 'CIC', 'MUC16', 'PIK3CA', 
    'NF1', 'PIK3R1', 'FUBP1', 'RB1', 'NOTCH1', 'BCOR', 'CSMD3', 
    'SMARCA4', 'GRIN2A', 'IDH2', 'FAT4', 'PDGFRA'
]

# 2. Process Mutations
# Transpose so samples are rows and genes are columns
weseq_mut_trans = weseq_mut.set_index('Gene_Name').transpose()

# Keep only the genes that match TCGA and convert to binary (1=mutated, 0=wildtype)
weseq_mut_aligned = weseq_mut_trans[genes_tcga].copy()
weseq_mut_binary = weseq_mut_aligned.notnull().astype(int)
weseq_mut_binary = weseq_mut_binary.reset_index().rename(columns={'index': 'CGGA_ID'})

# 3. Process Clinical Features to match TCGA encoding
# Grade: LGG (WHO II/III) -> 0, GBM (WHO IV) -> 1
weseq_clin['Grade_aligned'] = weseq_clin['Grade'].apply(lambda x: 1 if 'IV' in str(x) else 0)

# Gender: Male -> 0, Female -> 1
weseq_clin['Gender_aligned'] = weseq_clin['Gender'].map({'Male': 0, 'Female': 1})

# Age: Rename for consistency
weseq_clin['Age_at_diagnosis'] = weseq_clin['Age']

# 4. Merge Clinical and Mutation data on CGGA_ID
weseq_processed = pd.merge(
    weseq_clin[['CGGA_ID', 'Grade_aligned', 'Gender_aligned', 'Age_at_diagnosis']], 
    weseq_mut_binary, 
    on='CGGA_ID'
)

# 5. Final Formatting
weseq_final = weseq_processed.rename(columns={
    'Grade_aligned': 'Grade',
    'Gender_aligned': 'Gender'
})

# Ensure columns are in the correct order: ID, Clinical Features, then Genes
final_cols = ['CGGA_ID', 'Grade', 'Gender', 'Age_at_diagnosis'] + genes_tcga
weseq_final = weseq_final[final_cols]

# 6. Save the file
weseq_final.to_csv('weseq_processed_with_id.csv', index=False)
print("Processing complete. File saved as 'weseq_processed_with_id.csv'.")

Processing complete. File saved as 'weseq_processed_with_id.csv'.


/tmp/ipykernel_19188/2717541100.py:6: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,13,14,15,16,17,18,20,22,23,26,29,30,31,33,34,36,37,38,40,41,42,43,44,45,47,48,52,54,55,56,57,58,59,60,62,63,64,65,66,67,69,70,71,72,74,75,76,77,78,80,81,83,85,86,87,89,90,93,94,96,100,102,104,105,106,109,111,112,113,114,115,117,120,123,124,125,126,128,131,132,134,138,142,144,148,150,151,152,154,155,156,157,158,160,161,162,165,166,167,168,169,170,171,172,173,175,178,179,180,181,184,185,189,190,191,192,194,195,196,197,198,199,200,201,204,206,208,210,211,212,213,215,216,217,218,219,222,223,224,225,230,231,234,235,237,238,239,240,242,246,247,249,250,251,253,254,255,256,257,259,260,261,262,264,266,268,274,275,278,280,281,283,284,285) have mixed types. Specify dtype option on import or set low_memory=False.
  weseq_mut = pd.read_csv('weseq_286_Mutation.csv')
